# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


####  Run this cell to set up and start your interactive session.


In [2]:
%stop_session

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
There is no current session.


In [5]:
%idle_timeout 30
%glue_version 6.0
%worker_type G.1X
%number_of_workers 5

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.10 
Current idle_timeout is None minutes.
idle_timeout has been set to 30 minutes.
Setting Glue version to: 6.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5


In [8]:
%extra_jars s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar, s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar 
%extra_py_files s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-2.1.1.zip

Extra jars to be included:
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar
Extra py files to be included:
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-2.1.1.zip
s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar,s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar


In [35]:
#%%configure
#{
#  "--datalake-formats": "iceberg,delta",
#  "--additional-python-modules": "duckdb,pyarrow,shapely",
#  "--conf": {
#    "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
#    "spark.kryo.registrator": "com.esri.geoanalytics.KryoRegistrator",
#    "spark.plugins": "com.esri.geoanalytics.Plugin",
#    "spark.sql.extensions": "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension",
#    "spark.sql.catalog.spark_catalog": "org.apache.spark.sql.delta.catalog.DeltaCatalog",
#    "spark.sql.catalog.glue_catalog": "org.apache.iceberg.spark.SparkCatalog",
#    "spark.sql.catalog.glue_catalog.catalog-impl": "org.apache.iceberg.aws.glue.GlueCatalog",
#    "spark.sql.catalog.glue_catalog.warehouse": "s3://pske-prd-customerexperienceadhoc/spatial_analysis/"
#  }}

The following configurations have been updated: {'--datalake-formats': 'iceberg,delta', '--additional-python-modules': 'duckdb,pyarrow,shapely', '--conf': {'spark.serializer': 'org.apache.spark.serializer.KryoSerializer', 'spark.kryo.registrator': 'com.esri.geoanalytics.KryoRegistrator', 'spark.plugins': 'com.esri.geoanalytics.Plugin', 'spark.sql.extensions': 'org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,io.delta.sql.DeltaSparkSessionExtension', 'spark.sql.catalog.spark_catalog': 'org.apache.spark.sql.delta.catalog.DeltaCatalog', 'spark.sql.catalog.glue_catalog': 'org.apache.iceberg.spark.SparkCatalog', 'spark.sql.catalog.glue_catalog.catalog-impl': 'org.apache.iceberg.aws.glue.GlueCatalog', 'spark.sql.catalog.glue_catalog.warehouse': 's3://pske-prd-customerexperienceadhoc/spatial_analysis/'}}


In [10]:
%%configure
{
 "--conf": "spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin",
}

The following configurations have been updated: {'--conf': 'spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin'}


In [1]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window
from pyspark.sql.functions import expr
from awsglue import DynamicFrame
from pyspark.sql.functions import col, to_timestamp
import pyspark.sql.functions as F

# Initialize Spark session
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
# spark = SparkSession.builder.getOrCreate()
spark = SparkSession.builder \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.kryo.registrator", "com.esri.geoanalytics.KryoRegistrator") \
    .config("spark.plugins", "com.esri.geoanalytics.Plugin") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.glue_catalog.warehouse", "s3://pske-prd-datalake/") \
    .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog") \
    .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .getOrCreate()
job = Job(glueContext)

# Check active session configs
print("Spark Extensions:", spark.conf.get("spark.sql.extensions", "None"))
print("Serializer:", spark.conf.get("spark.serializer", "None"))
print("Spark Session successfully instantiated!")


Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 30
Session ID: 298d6562-2b3a-447a-9cd5-05321d36e0bf
Applying the following default arguments:
--glue_kernel_version 1.0.10
--enable-glue-datacatalog true
--conf spark.serializer=org.apache.spark.serializer.KryoSerializer --conf spark.kryo.registrator=com.esri.geoanalytics.KryoRegistrator --conf spark.plugins=com.esri.geoanalytics.Plugin
--extra-py-files s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-2.1.1.zip
--extra-jars s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics_2.13-2.1.1.jar,s3://pske-prd-customerexperienceadhoc/Geo_Analytics_Engine_License/lib_files/geoanalytics-natives_2.13-2.1.1.jar
Waiting for session 298d6562-2b3a-447a-9cd5-05321d36e0bf to get into ready status...
Session 298d6562-2b3a-447a-9cd5-05321d36e0bf has been created.
Spark Extensions: com.esri.geoanalytics.sq

In [2]:
import geoanalytics
from geoanalytics.sql import functions as ST
# Authenticate with the license file
geoanalytics.auth(username = "PTL_GAE",password = "MICXA_1008-10!")

In [3]:
# global state Filter
STATE = "DC"
STATE_LOWER = STATE.lower()
print(f"Global state filter set to: {STATE}")

Global state filter set to: DC


In [4]:
df_polk = glueContext.create_data_frame.from_catalog(
    database="ptl_marketuniverse",
    table_name="mu_polk",
    additional_options={"datalake_formats": "iceberg"}
)
print("Iceberg Polk table loaded successfully via GlueContext.")

df_rigdig = glueContext.create_data_frame.from_catalog(
    database="ptl_marketuniverse",
    table_name="mu_rigdig",
    additional_options={"datalake_formats": "iceberg"}
)
print("Iceberg RigDig table loaded successfully via GlueContext.")

df_mkt_analytics = glueContext.create_data_frame.from_catalog(
    database="ptl_marketuniverse",
    table_name="marketing_analytics",
    additional_options={"datalake_formats": "iceberg"}
)
print("Iceberg marketing_analytics table loaded successfully via GlueContext.")

df_dnb_master = spark.read.table("ptl_marketuniverse.mu_dnb_data_master")
print("Delta D&B Master table loaded successfully via Spark.")
df_mkt_analytics.select("final_duns_number").show(1)

Iceberg Polk table loaded successfully via GlueContext.
Iceberg RigDig table loaded successfully via GlueContext.
Iceberg marketing_analytics table loaded successfully via GlueContext.
Delta D&B Master table loaded successfully via Spark.
+-----------------+
|final_duns_number|
+-----------------+
|        001000678|
+-----------------+
only showing top 1 row
/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.


In [5]:
df_dnb_pts = df_dnb_master.filter(
    F.col("latitude").isNotNull() & F.col("longitude").isNotNull()
).withColumn(
    "geometry",
    ST.point("longitude", "latitude", 4326)
)

# 2. Read GeoJSON boundary from S3
s3_geojson_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/exports/dnb_parcel_match_dc_geopqt/DC_district_Bndy.geojson"

df_boundary = (
    spark.read.format("geojson")
    .load(s3_geojson_path)
    .filter(F.col("district") == "0860 WASHINGTON DC")
)

# 3. Perform Spatial Join using ST.contains
df_dnb_dc = df_dnb_pts.join(
    df_boundary,
    ST.contains(df_boundary["geometry"], df_dnb_pts["geometry"]),
    how="inner"
).select(df_dnb_pts["*"])


# 4. Preview DC Filtered Market Universe
df_dnb_dc.select("duns_number", "business_name", "latitude", "longitude").show(5, truncate=False)
print(f"Total D&B Master Records in District 0860 WASHINGTON DC: {df_dnb_dc.count():,}") 

+-----------+-----------------------+----------+-----------+
|duns_number|business_name          |latitude  |longitude  |
+-----------+-----------------------+----------+-----------+
|031827911  |SANEL CONST GROUP INC  |+39.140417|-077.054341|
|065361921  |ZONUM INC              |+39.135556|-077.046218|
|076034001  |JEHEMY CONSTRUCTION INC|+39.140417|-077.054341|
|111848155  |KRAMER ENTERPRISES, LLC|+39.140417|-077.054341|
|117303331  |J & JAY'S TRANSPORT LLC|+39.140943|-077.058337|
+-----------+-----------------------+----------+-----------+
only showing top 5 rows
Total D&B Master Records in District 0860 WASHINGTON DC: 161,537


In [6]:
df_boundary.count()

1


In [10]:
A = df_dnb_dc.alias("A")
P = df_polk.alias("P")
R = df_rigdig.alias("R")
M = df_mkt_analytics.alias("M")
mu_df = (
    A
    .join(F.broadcast(P), on="duns_number", how="left")
    .join(F.broadcast(R), on="duns_number", how="left")
    .join(F.broadcast(M), F.col("A.duns_number") == F.col("M.site_duns_number"), how="left")
    .select(
        # Company Identification & Linkage
        F.col("A.duns_number"),
        F.col("M.final_duns_number"),
        F.col("A.business_name"),
        F.col("A.parent_duns_number"),
        F.col("A.headquarter_duns_number"),
        F.col("A.dot_linkage"),
        F.col("A.global_ultimate_duns_number"),
        F.col("A.global_ultimate_indicator"),
        F.col("A.global_ultimate_business_name"),
        F.col("A.out_of_business_indicator"),
        F.col("A.duns_linkage"),
        F.col("A.number_of_family_members"),

        # General Industry & Classification
        F.col("A.penske_category"),
        F.col("A.tradestyle_name"),
        F.col("A.line_of_business"),
        F.col("A.formatted_dot_linkage"),
        F.col("A.us_1987_sic_1"),
        F.col("A.naics"),

        # Location & Address Fields
        F.col("A.street_address"),
        F.col("A.city_name"),
        F.col("A.state_province_abbr"),
        F.col("A.postal_code"),
        F.col("A.county_name"),
        F.col("A.latitude"),
        F.col("A.longitude"),

        # Metrics & Financials
        F.col("A.employees_total"),
        F.col("A.employees_here"),
        F.col("A.sales_volume_us_dollars"),
        F.col("A.telephone_number"),

        # Contact Information
        F.col("A.chief_exec_officer_full_name"),
        F.col("A.chief_exec_officer_title"),
        F.col("A.first_executive_first_name"),
        F.col("A.first_executive_last_name"),
        F.col("A.first_executive_title"),

        # Refined 7-Tier Sales Target Segment Classification
        F.when(F.coalesce(F.col("A.sales_volume_us_dollars"), F.lit(0)) >= 50000000, "1. Mega ($50M+)")
         .when(F.coalesce(F.col("A.sales_volume_us_dollars"), F.lit(0)) >= 10000000, "2. Large ($10M - $50M)")
         .when(F.coalesce(F.col("A.sales_volume_us_dollars"), F.lit(0)) >= 5000000, "3. Upper Mid ($5M - $10M)")
         .when(F.coalesce(F.col("A.sales_volume_us_dollars"), F.lit(0)) >= 2500000, "4. Lower Mid ($2.5M - $5M)")
         .when(F.coalesce(F.col("A.sales_volume_us_dollars"), F.lit(0)) >= 1000000, "5. Small ($1M - $2.5M)")
         .when(F.coalesce(F.col("A.sales_volume_us_dollars"), F.lit(0)) >= 250000, "6. Micro ($250K - $1M)")
         .when(F.col("A.sales_volume_us_dollars").isNull(), "8. Unknown (No Data)")
         .otherwise("7. Nano / Pre-Rev (<$250K)")
         .alias("sales_target_segment"),

        # Polk Vehicle Registration Enrichment
        F.col("P.confidence_code").alias("polk_confidence_code"),
        F.col("P.total_fleet_size_gvw_3_8"),

        # RigDig Enrichment
        F.col("R.confidence_code").alias("rigdig_confidence_code"),
        F.col("R.ent_usdot_total_pwr"),
        F.col("R.eqt_class_3to8_units"),
        F.col("R.eqt_class_all_units")
    )
)

# 5. Filter for target State (DC) to construct dc_mu_df
#dc_mu_df = mu_df.filter(F.col("state_province_abbr") == STATE)

# 6. Execute Counts and Breakdown
dc_mu_df = mu_df
print(f"Total Market Universe Companies ({STATE}): {dc_mu_df.count()}")

print("\nRefined Breakdown by Sales Target Segment:")
(
    dc_mu_df
    .groupBy("sales_target_segment")
    .count()
    .orderBy("sales_target_segment")
    .show(truncate=False)
)

Total Market Universe Companies (DC): 167199

Refined Breakdown by Sales Target Segment:
+--------------------------+------+
|sales_target_segment      |count |
+--------------------------+------+
|1. Mega ($50M+)           |1206  |
|2. Large ($10M - $50M)    |2942  |
|3. Upper Mid ($5M - $10M) |3629  |
|4. Lower Mid ($2.5M - $5M)|4300  |
|5. Small ($1M - $2.5M)    |7715  |
|6. Micro ($250K - $1M)    |22011 |
|7. Nano / Pre-Rev (<$250K)|125396|
+--------------------------+------+


In [11]:
# apply is_child logic
dc_mu_df = dc_mu_df.withColumn(
    "is_child",
    F.when(
        F.col("duns_linkage").contains("|") | F.col("dot_linkage").contains("."), 
        "Yes"
    ).otherwise("No")
)
print("child defined")

child defined


In [ ]:
dc_mu_df.filter("number_of_family_members <=1").count()

Execution Interrupted. Attempting to cancel the statement (statement_id=14)


In [12]:
# Read LSR report from Territory analysis
dc_prospects_df = spark.read.format("csv")\
                            .option("header", "true")\
                            .option("inferSchema", "true")\
                            .load("s3://pske-prd-customerexperienceadhoc/spatial_analysis/lsr_imports/LSR_DC_SF_Data.csv")

dc_prospects_df =  dc_prospects_df.toDF(*[col.lower() for col in dc_prospects_df.columns])

dc_prospects_df = dc_prospects_df.drop("STREET_ADDRESS")
print(dc_prospects_df.count())
dc_prospects_df.groupBy("sf_account_id").count().show(truncate=False)
dc_prospects_df = dc_prospects_df.drop("BUSINESS_NAME")

56498
+-------------+-----+
|sf_account_id|count|
+-------------+-----+
|No           |54224|
|FIELD        |1    |
|Yes          |2273 |
+-------------+-----+


In [14]:
# Filter out the sf_account_id (sales force account not present)
dc_ps_join = dc_mu_df.join(dc_prospects_df,"duns_number","inner") #.where(col("sf_account_id") == "No")
#dc_ps_join =  dc_ps_join.filter(col("state") == STATE)
print(f"{STATE} prospect count: {dc_ps_join.count()}")


DC prospect count: 58787


In [ ]:
dc_ps_join.groupBy("sf_account_id").count().show(truncate=False)
dc_ps_join.printSchema()
dc_ps_join.show(1)


In [9]:
dc_ps_join.groupBy("state_province_abbr").count().show(truncate=False)

NameError: name 'dc_ps_join' is not defined


In [20]:
from pyspark.sql import functions as F
from functools import reduce

# Map state abbreviations to S3 folder names
states_map = {
    "DC": "DC",
    "MD": "MD", # Update folder name if MD folder is named 'Maryland'
    "VA": "VA"  # Update folder name if VA folder is named 'Virginia'
}

base_s3_path = "s3://Regrid_Parcels/State_Level_US_Parcels"

state_all_dfs = []
state_non_res_dfs = []

# Define standard Non-Residential filter condition
# Non-residential zoning OR Non-residential land use (LBCS)
non_res_condition = (
    (F.col("zoning_typ") != "Residential") |
    F.col("lbcs_activ").isNull() |
    ~(
        (F.col("lbcs_activ").cast("int").between(1000, 1999)) |
        (F.col("lbcs_act_1").ilike("%Household%"))
    )
)

for state_code, folder in states_map.items():
    s3_path = f"{base_s3_path}/{folder}/"
    
    # Load state parcels and assign state tag
    df_state = spark.read.parquet(s3_path).withColumn("source_state", F.lit(state_code))
    
    # Filter for non-residential parcels for this state
    df_state_non_res = df_state.filter(non_res_condition)
    
    # Calculate counts
    total_cnt = df_state.count()
    non_res_cnt = df_state_non_res.count()
    
    # Specific metric counts for logging
    non_res_zoning_cnt = df_state.filter("zoning_typ != 'Residential'").count()
    non_res_landuse_cnt = df_state.filter(
        F.col("lbcs_activ").isNull() |
        ~(
            (F.col("lbcs_activ").cast("int").between(1000, 1999)) |
            (F.col("lbcs_act_1").ilike("%Household%"))
        )
    ).count()
    
    print(f"--- {state_code} Parcel Summary ---")
    print(f"  Total {state_code} counts: {total_cnt:,}")
    print(f"  Non-Residential Zoning counts: {non_res_zoning_cnt:,}")
    print(f"  Non-Residential Landuse counts: {non_res_landuse_cnt:,}")
    print(f"  Combined Non-Residential {state_code} counts: {non_res_cnt:,}\n")
    
    state_all_dfs.append(df_state)
    state_non_res_dfs.append(df_state_non_res)

# 1. Combine ALL state DataFrames into a single unified DataFrame
parcels_all_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), state_all_dfs)

# 2. Combine NON-RESIDENTIAL state DataFrames into a single unified DataFrame
non_res_parcels_all_df = reduce(lambda df1, df2: df1.unionByName(df2, allowMissingColumns=True), state_non_res_dfs)

print(f"Combined Tri-State All Parcels Count: {parcels_all_df.count():,}")
print(f"Combined Tri-State Non-Residential Parcels Count: {non_res_parcels_all_df.count():,}")

--- DC Parcel Summary ---
  Total DC counts: 207,339
  Non-Residential Zoning counts: 34,068
  Non-Residential LBCS counts: 22,525
  Unique Non-Residential DC counts: 46,342

--- MD Parcel Summary ---
  Total MD counts: 2,489,839
  Non-Residential Zoning counts: 713,696
  Non-Residential LBCS counts: 185,448
  Unique Non-Residential MD counts: 675,001

--- VA Parcel Summary ---
  Total VA counts: 4,240,836
  Non-Residential Zoning counts: 2,321,338
  Non-Residential LBCS counts: 217,064
  Unique Non-Residential VA counts: 1,713,675

Combined Tri-State All Parcels Count: 6,938,014
Combined Tri-State Non-Residential Parcels Count: 2,435,018


In [8]:
dnb_df_parcel_intersect.printSchema()

NameError: name 'dnb_df_parcel_intersect' is not defined


In [29]:
# select parcel and filter zoing_typ="Residential"
#parcel_src = spark.read.parquet("s3://pske-prd-customerexperienceadhoc/Regrid_Parcels/State_Level_US_Parcels/DC/")
#print(f" Total DC counts {parcel_src.count()}")
#print(f""" Non Residential DC counts {parcel_src.filter("zoning_typ != 'Residential'").count()}""")
#print(f""" Landuse code not residential {parcel_src.filter(F.col("lbcs_activ").isNull() |
#    ~(
#        (F.col("lbcs_activ").cast("int").between(1000, 1999)) |
#        (F.col("lbcs_act_1").ilike("%Household%")) 
#    )).count()}""")


 Total DC counts 207339
 Non Residential DC counts 34068
 Landuse code not residential 34361


In [ ]:
# Configuration & Paths
S3_TEMP_BASE = "s3://pske-prd-customerexperienceadhoc/Regrid_Parcels/temp_stage_output"
BASE_S3_PATH = "s3://pske-prd-customerexperienceadhoc/Regrid_Parcels/State_Level_US_Parcels"

In [49]:
import geoanalytics
from geoanalytics.sql import functions as ST
from pyspark.sql import functions as F
from pyspark.sql.window import Window


STATE_CODE = "DC"
FOLDER = "DC"

print(f"=== STARTING {STATE_CODE} PROCESSING ===")

# 1. Read State Parcels
s3_path = f"{BASE_S3_PATH}/{FOLDER}/"
df_state = spark.read.parquet(s3_path).withColumn("source_state", F.lit(STATE_CODE))

# 2. Filter Non-Residential Parcels
non_res_condition = (
    (F.col("zoning_typ") != "Residential") |
    F.col("lbcs_activ").isNull() |
    ~(
        (F.col("lbcs_activ").cast("int").between(1000, 1999)) |
        (F.col("lbcs_act_1").ilike("%Household%"))
    )
)

df_state_non_res = df_state.filter(non_res_condition).dropDuplicates(["parcelnu_1"])

# 3. Parcel Geometry Processing
parcel_nonres_geom = df_state_non_res.withColumn(
    "polygeom", ST.geom_from_binary("geometry", sr=4326)
).select(
    "polygeom", "parcelnu_1", "usecode", "zoning", "zoning_des", 
    "zoning_typ", "zoning_sub", "lbcs_activ", "owner", "usedesc"
)
# Register geometry field for spatial optimization
parcel_nonres_geom = parcel_nonres_geom.st.set_geometry_field("polygeom")

# 4. Filter Combined dc_ps_join for Target State
dnb_df_geom = (
    dc_ps_join
    .filter(F.col("state_province_abbr") == STATE_CODE)
    .withColumn("pointgeom", ST.point("longitude", "latitude", sr=4326))
)
dnb_df_geom = dnb_df_geom.st.set_geometry_field("pointgeom")

# 5. Spatial Join (Using native GeoAnalytics spatial execution)
dnb_parcel_intersect = dnb_df_geom.join(
    parcel_nonres_geom,
    ST.contains(parcel_nonres_geom["polygeom"], dnb_df_geom["pointgeom"]),
    how="left"
)

# 6. Window Deduplication
dnb_parcel_intersect = dnb_parcel_intersect.withColumn(
    "parcel_flag", F.when(F.col("parcelnu_1").isNotNull(), "Y").otherwise("N")
)

no_dup_window = Window.partitionBy("duns_number").orderBy(
    F.col("parcel_flag").desc(), F.col("lbcs_activ")
)

# Preserve spatial point geometry as shape column
no_dup_df = (
    dnb_parcel_intersect
    .drop("polygeom")
    .withColumnRenamed("pointgeom", "shape")
    .withColumn("rn", F.row_number().over(no_dup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)
no_dup_df = no_dup_df.st.set_geometry_field("shape")

# 7. MATERIALIZATION STEP: Write to Parquet Once to Break Lineage
parquet_path = f"{S3_TEMP_BASE}/parquet/dedup_{STATE_CODE}/"
no_dup_df.write.mode("overwrite").parquet(parquet_path)

# 8. Read back materialized data for lightweight downstream exports
staged_df = spark.read.parquet(parquet_path).st.set_geometry_field("shape")

# CSV Export (Drop geometry binary field for tabular writing)
csv_path = f"{S3_TEMP_BASE}/csv/dedup_{STATE_CODE}/"
staged_df.drop("shape").drop("geometry").write.mode("overwrite").option("header", "true").option("quoteAll", "true").csv(csv_path)


# Direct count from materialized dataframe (Fast execution)
record_count = staged_df.count()
print(f"=== {STATE_CODE} COMPLETED: {record_count:,} records written (Parquet, CSV, & GeoParquet) ===\n")

=== STARTING DC PROCESSING ===
=== DC COMPLETED: 4,298 records written (Parquet, CSV, & GeoParquet) ===


In [50]:
# GeoParquet Export
geoparquet_path = f"{S3_TEMP_BASE}/geoparquet/dedup_{STATE_CODE}/"
staged_df.drop("geometry").write.format("geoparquet").mode("overwrite").save(geoparquet_path)

In [ ]:
STATE_CODE = "MD"

print(f"=== STARTING {STATE_CODE} PROCESSING ===")

df_state = spark.read.parquet(f"{BASE_S3_PATH}/{STATE_CODE}/")
    .select(
        "geometry", "parcelnu_1", "usecode", "zoning", "zoning_des", 
        "zoning_typ", "zoning_sub", "lbcs_activ", "lbcs_act_1", "owner", "usedesc"
    )
    .withColumn("source_state", F.lit(STATE_CODE))

non_res_condition = (
    (F.col("zoning_typ") != "Residential") |
    F.col("lbcs_activ").isNull() |
    ~(
        (F.col("lbcs_activ").cast("int").between(1000, 1999)) |
        (F.col("lbcs_act_1").ilike("%Household%"))
    )
)

df_state_non_res = df_state.filter(non_res_condition).dropDuplicates(["parcelnu_1"])

parcel_nonres_geom = df_state_non_res.withColumn(
    "polygeom", ST.geom_from_binary("geometry", sr=4326)
).select(
    "polygeom", "parcelnu_1", "usecode", "zoning", "zoning_des", 
    "zoning_typ", "zoning_sub", "lbcs_activ", "owner", "usedesc"
)

dnb_df_geom = (
    dc_ps_join
    .filter(F.col("state_province_abbr") == STATE_CODE)
    .withColumn("pointgeom", ST.point("longitude", "latitude", sr=4326))
)
dnb_df_geom = dnb_df_geom.st.set_geometry_field("pointgeom")
# Filter points that fall strictly inside the parcel bounding box
dnb_bounded_points = dnb_df_geom.filter(ST.env_intersects(dnb_df_geom["pointgeom"], F.lit(parcel_bbox)))

dnb_parcel_intersect = dnb_bounded_points.join(
    parcel_nonres_geom,
    ST.contains(parcel_nonres_geom["polygeom"], dnb_bounded_points["pointgeom"]),
    how="left"
)

dnb_parcel_intersect = dnb_parcel_intersect.withColumn(
    "parcel_flag", F.when(F.col("parcelnu_1").isNotNull(), "Y").otherwise("N")
)

no_dup_window = Window.partitionBy("duns_number").orderBy(
    F.col("parcel_flag").desc(), F.col("lbcs_activ")
)

no_dup_df = (
    dnb_parcel_intersect
    .drop("shape", "polygeom", "pointgeom")
    .withColumn("rn", F.row_number().over(no_dup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Write Parquet
#parquet_path = f"{S3_TEMP_BASE}/parquet/dedup_{STATE_CODE}/"
#no_dup_df.write.mode("overwrite").parquet(parquet_path)

# Write CSV
csv_path = f"{S3_TEMP_BASE}/csv/dedup_{STATE_CODE}/"
no_dup_df.write.mode("overwrite").option("header", "true").option("quoteAll", "true").csv(csv_path)
no_dup_df.write.format("geoparquet").mode("overwrite").save(geoparquet_path)

md_final_df = spark.read.parquet(parquet_path)
print(f"=== {STATE_CODE} COMPLETED: {md_final_df.count():,} records written (Parquet & CSV) ===\n")

Exception encountered while retrieving session: Error when retrieving credentials from iam-role: Credential refresh failed, response did not contain: access_key, secret_key, token, expiry_time 
Traceback (most recent call last):
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/aws_glue_interactive_sessions_kernel/glue_kernel_base/BaseKernel.py", line 726, in get_current_session
    current_session = self.kernel_gateway.get_session(self.get_session_id())["Session"]
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/aws_glue_interactive_sessions_kernel/glue_kernel_utils/KernelGateway.py", line 194, in get_session
    return self.glue_client.get_session(Id=session_id)
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/client.py", line 602, in _api_call
    return self._make_api_call(operation_name, kwargs)
  File "/home/jupyter-user/.local/lib/python3.9/site-packages/botocore/context.py", line 123, in wrapper
    return func(*args, **kwargs)
  F

In [ ]:
STATE_CODE = "VA"

print(f"=== STARTING {STATE_CODE} PROCESSING ===")

df_state = spark.read.parquet(f"{BASE_S3_PATH}/{STATE_CODE}/").withColumn("source_state", F.lit(STATE_CODE))

non_res_condition = (
    (F.col("zoning_typ") != "Residential") |
    F.col("lbcs_activ").isNull() |
    ~(
        (F.col("lbcs_activ").cast("int").between(1000, 1999)) |
        (F.col("lbcs_act_1").ilike("%Household%"))
    )
)

df_state_non_res = df_state.filter(non_res_condition).dropDuplicates(["parcelnu_1"])

parcel_nonres_geom = df_state_non_res.withColumn(
    "polygeom", ST.geom_from_binary("geometry", sr=4326)
).select(
    "polygeom", "parcelnu_1", "usecode", "zoning", "zoning_des", 
    "zoning_typ", "zoning_sub", "lbcs_activ", "owner", "usedesc"
)

dnb_df_geom = (
    dc_ps_join
    .filter(F.col("state_province_abbr") == STATE_CODE)
    .withColumn("pointgeom", ST.point("longitude", "latitude", sr=4326))
)
dnb_df_geom = dnb_df_geom.st.set_geometry_field("pointgeom")

dnb_parcel_intersect = dnb_df_geom.join(
    F.broadcast(parcel_nonres_geom),
    ST.contains(parcel_nonres_geom["polygeom"], dnb_df_geom["pointgeom"]),
    how="left"
)

dnb_parcel_intersect = dnb_parcel_intersect.withColumn(
    "parcel_flag", F.when(F.col("parcelnu_1").isNotNull(), "Y").otherwise("N")
)

no_dup_window = Window.partitionBy("duns_number").orderBy(
    F.col("parcel_flag").desc(), F.col("lbcs_activ")
)

no_dup_df = (
    dnb_parcel_intersect
    .drop("shape", "polygeom", "pointgeom")
    .withColumn("rn", F.row_number().over(no_dup_window))
    .filter(F.col("rn") == 1)
    .drop("rn")
)

# Write Parquet
parquet_path = f"{S3_TEMP_BASE}/parquet/dedup_{STATE_CODE}/"
no_dup_df.write.mode("overwrite").parquet(parquet_path)

# Write CSV
csv_path = f"{S3_TEMP_BASE}/csv/dedup_{STATE_CODE}/"
no_dup_df.write.mode("overwrite").option("header", "true").option("quoteAll", "true").csv(csv_path)
no_dup_df.write.format("geoparquet").mode("overwrite").save(geoparquet_path)

va_final_df = spark.read.parquet(parquet_path)
print(f"=== {STATE_CODE} COMPLETED: {va_final_df.count():,} records written (Parquet & CSV) ===\n")

In [30]:
df_zone_nonres = parcel_src.filter(F.col("zoning_typ").isNull() | (F.col("zoning_typ") != "Residential"))
df_lbcs_nonres = parcel_src.filter(F.col("zoning_typ").isNull() | ~ (F.col("lbcs_activ").cast("int").between(1000, 1999)))
parcel_nonres = df_zone_nonres.unionByName(df_lbcs_nonres).dropDuplicates(["parcelnu_1"])
parcel_nonres.count()

46342


In [48]:
anti_parcels = parcel_src.join(parcel_nonres, "parcelnu_1", "left_anti")
anti_parcels.count()
anti_parcels.groupBy("lbcs_activ", "zoning_typ").count().show()

+----------+-----------+------+
|lbcs_activ| zoning_typ| count|
+----------+-----------+------+
|      NULL|Residential|  8343|
|    1100.0|Residential|152519|
|    1200.0|Residential|    60|
|    1300.0|Residential|    54|
+----------+-----------+------+


In [21]:
parcel_nonres_geom = non_res_parcels_all_df.withColumn("polygeom", ST.geom_from_binary("geometry", sr=4326))
parcel_nonres_geom = parcel_nonres_geom.select("polygeom","parcelnu_1","usecode","zoning","zoning_des","zoning_typ","zoning_sub","lbcs_activ","owner","usedesc","zoning_des")
print(parcel_nonres_geom.count())
parcel_nonres_geom.show(5)
parcel_nonres_geom.printSchema()

2435018
+--------------------+--------------+-------+------+--------------------+----------+----------+----------+--------------------+-----------+--------------------+
|            polygeom|    parcelnu_1|usecode|zoning|          zoning_des|zoning_typ|zoning_sub|lbcs_activ|               owner|    usedesc|          zoning_des|
+--------------------+--------------+-------+------+--------------------+----------+----------+----------+--------------------+-----------+--------------------+
|{"rings":[[[-76.9...|              |       | PDR-1|Production Distri...|   Special|   Special|      NULL|                    |           |Production Distri...|
|{"rings":[[[-76.9...|0000Unassessed|       |    UZ|             Unzoned|   Special|   Special|      NULL|                    |           |             Unzoned|
|{"rings":[[[-77.0...|      00010843|    191|    UZ|             Unzoned|   Special|   Special|      NULL|UNITED STATES OF ...|Vacant-True|             Unzoned|
|{"rings":[[[-77.0...|    

In [22]:
# Create geometry for DC
dnb_df_geom = dc_ps_join.withColumn("pointgeom", ST.point("longitude","latitude", sr=4326))
dnb_df_geom = dnb_df_geom.st.set_geometry_field("pointgeom")
#dnb_df_geom.show(2, truncate = 90)
dnb_df_geom.select("latitude","longitude","pointgeom").show(2, truncate = 90)

+----------+-----------+------------------------------+
|  latitude|  longitude|                     pointgeom|
+----------+-----------+------------------------------+
|+39.140417|-077.054341|{"x":-77.054341,"y":39.140417}|
|+39.135556|-077.046218|{"x":-77.046218,"y":39.135556}|
+----------+-----------+------------------------------+
only showing top 2 rows


In [ ]:
#parcel_nonres_geom.count()
dnb_df_geom.count()
# count previous run 09/03 - 4337
#dnb_df_geom.printSchema()
#parcel_nonres_geom.printSchema()
#print(parcel_nonres_geom.schema["polygeom"].dataType)
#print(dnb_df_geom.schema["pointgeom"].dataType)
#dnb_df_geom.select(ST.srid(F.col("pointgeom"))).distinct().show()
#parcel_nonres_geom.select(ST.srid(F.col("polygeom"))).distinct().show()

In [24]:
dnb_df_parcel_intersect = dnb_df_geom.join(F.broadcast(parcel_nonres_geom),
    ST.contains(parcel_nonres_geom["polygeom"],dnb_df_geom["pointgeom"]), "left")
#.withColumn("parcel_non_resi_flag",F.lit("Y"))
#dnb_df_parcel_intersect.groupBy("parcel_non_resi_flag").count().show()
print("spatial join finished")

spatial join finished


In [27]:
dnb_df_parcel_intersect = dnb_df_parcel_intersect.withColumn("parcel_flag", F.when(F.col("parcelnu_1").isNotNull(),"Y").otherwise("N"))
#summary_df = dnb_df_parcel_intersect.coalesce(10).groupBy("parcel_flag").count()
#summary_df.show()
# previous Y 14591 N 2029 I reversed parcel flag

In [ ]:
dnb_df_parcel_intersect.printSchema()

In [30]:
no_dup_window = Window.partitionBy("duns_number").orderBy(F.col("parcel_flag").desc(),F.col("lbcs_activ"))
no_dup_df = dnb_df_parcel_intersect.drop("shape").withColumn("rn", F.row_number().over(no_dup_window)).filter(F.col("rn")==1).drop("rn")
print(no_dup_df.count())

Py4JJavaError: An error occurred while calling o1239.count.
: org.apache.spark.SparkException: Job aborted due to stage failure: Total size of serialized results of 8 tasks (1122.5 MiB) is bigger than spark.driver.maxResultSize (1024.0 MiB)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$3(DAGScheduler.scala:3620)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2(DAGScheduler.scala:3620)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$abortStage$2$adapted(DAGScheduler.scala:3612)
	at scala.collection.immutable.List.foreach(List.scala:323)
	at org.apache.spark.scheduler.DAGScheduler.abortStage(DAGScheduler.scala:3612)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1(DAGScheduler.scala:1397)
	at org.apache.spark.scheduler.DAGScheduler.$anonfun$handleTaskSetFailed$1$adapted(DAGScheduler.scala:1397)
	at scala.Option.foreach(Option.scala:437)
	at org.apache.spark.scheduler.DAGSchedule

In [ ]:
#.groupBy("lbcs_activ", "lbcs_act_1") \
#    .agg(
#        F.countDistinct("duns_number").alias("distinct_duns_count"),
#        F.count("ll_uuid").alias("total_matched_parcels")
#    ).orderBy(F.col("distinct_duns_count").desc())
# remaining_distribution.show(50, truncate=False)

In [ ]:
no_dup_df.groupBy("parcel_flag").count().show()

In [50]:
no_dup_df = no_dup_df.drop("polygeom")
no_dup_df = no_dup_df.withColumnRenamed("pointgeom", "shape")
no_dup_df.printSchema()

root
 |-- duns_number: string (nullable = true)
 |-- business_name: string (nullable = true)
 |-- parent_duns_number: string (nullable = true)
 |-- headquarter_duns_number: string (nullable = true)
 |-- dot_linkage: string (nullable = true)
 |-- global_ultimate_duns_number: string (nullable = true)
 |-- global_ultimate_indicator: string (nullable = true)
 |-- global_ultimate_business_name: string (nullable = true)
 |-- out_of_business_indicator: string (nullable = true)
 |-- duns_linkage: string (nullable = true)
 |-- number_of_family_members: long (nullable = true)
 |-- penske_category: string (nullable = true)
 |-- tradestyle_name: string (nullable = true)
 |-- line_of_business: string (nullable = true)
 |-- formatted_dot_linkage: string (nullable = true)
 |-- us_1987_sic_1: string (nullable = true)
 |-- naics: string (nullable = true)
 |-- street_address: string (nullable = true)
 |-- city_name: string (nullable = true)
 |-- state_province_abbr: string (nullable = true)
 |-- postal_

In [52]:
geoparquet_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/exports/dnb_parcel_match_dc_geopqt"
no_dup_df.drop("zoning_des").coalesce(1).write \
                .format("geoparquet") \
                .mode("overwrite") \
                .option("compression", "snappy") \
                .save(geoparquet_s3_path)
print("successfully Exported")

successfully Exported


In [49]:
# Write out as a single CSV file with headers
csv_export_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/exports/dnb_parcel_match_dc/"
no_dup_df.drop("shape").drop("pointgeom").drop("polygeom").coalesce(1).write \
    .format("csv") \
    .mode("overwrite") \
    .option("header", "true") \
    .option("emptyValue", "") \
    .save(csv_export_s3_path)

print("csv exported")

csv exported


In [19]:
geoparquet_s3_path = "s3://pske-prd-customerexperienceadhoc/spatial_analysis/dnb_parcel_match_dc"
dc_pqt = spark.read.parquet(geoparquet_s3_path)
print(dc_pqt.count())

AnalysisException: [PATH_NOT_FOUND] Path does not exist: s3://pske-prd-customerexperienceadhoc/spatial_analysis/dnb_parcel_match_dc.


In [7]:

df_with_geom = dc_pqt.withColumn("shape", ST.geom_from_binary("shape", sr=4326))                    

df_with_geom.select("shape").printSchema()

root
 |-- shape: geometry (nullable = true)


In [28]:
import requests
import geoanalytics
import urllib3

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

portal_url = "https://gisstgportal.penske.com/portal"
oauth_url = f"{portal_url}/sharing/rest/oauth2/token"

# 1. Get OAuth2 Token using App ID and Secret
payload = {
    "client_id": "Vq99z7jaCI13Wapo",
    "client_secret": "d37ae7cd431c4e90a81edce3edea79a6",
    "grant_type": "client_credentials",
    "expiration": 120,
    "f": "json"
}

res = requests.post(oauth_url, data=payload, verify=False).json()

if "access_token" in res:
    oauth_token = res["access_token"]
    print("OAuth Access Token acquired successfully!")
    
    # 2. Register GIS with the OAuth Access Token
    geoanalytics.register_gis(
        name="myGIS",
        url=portal_url,
        token=oauth_token,
        verify_cert=False
    )
else:
    print("OAuth Error:", res)

ConnectionError: HTTPSConnectionPool(host='gisstgportal.penske.com', port=443): Max retries exceeded with url: /portal/sharing/rest/oauth2/token (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7fe89a58b650>: Failed to establish a new connection: [Errno -2] Name or service not known'))


In [25]:
import geoanalytics
#geoanalytics.register_gis("myPortal", "https://gisstgportal.penske.com/portal", username="svc_penske_aws_stg", password="Pen35!DWThj2026")
geoanalytics.register_gis("myGIS1", "https://gisstgserver.penske.com/arcgis", username="svc_penske_aws_stg", password="Pen35!DWThj2026")


IllegalArgumentException: GIS login failed.


In [12]:
myFS="https://services.arcgis.com/P3ePLMYs2RVChkJx/ArcGIS/rest/services/World_Cities/FeatureServer/0"
myFSDataFrame = spark.read.format('feature-service').load(myFS)
group = myFSDataFrame.selectExpr("CNTRY_NAME", "POP").groupBy("CNTRY_NAME").avg("POP")
group.where("avg(POP) > 50000").show()

+-------------+------------------+
|   CNTRY_NAME|          avg(POP)|
+-------------+------------------+
|    Nicaragua|           74500.0|
|     Cameroon|          682800.0|
|        Congo|          315500.0|
|       Israel|          308123.5|
|    Indonesia|1148963.4814814816|
|      Myanmar|          583645.8|
|  South Korea|         2215724.9|
|    Australia|1147753.9333333333|
|French Guiana|           57614.0|
| Burkina Faso|126275.86206896552|
|        Ghana|          374058.1|
|         Togo|          209891.5|
|  The Bahamas|          266100.0|
|       Turkey| 408656.7164179105|
|      Armenia|         1079732.0|
|      Somalia|158411.76470588235|
|       Sweden| 66458.33333333333|
|      Croatia|          459320.5|
|  Afghanistan| 142620.6896551724|
|     Thailand|506188.81944444444|
+-------------+------------------+
only showing top 20 rows


In [9]:
# write to portal layer
geoanalytics.register_gis("myGIS", username="svc_penske_aws_stg", password="Pen35!DWThj2026")
service_name = "dnb_prospects"
portal_url = "https://gisstgserver.penske.com/arcgis" 

df_with_geom.write \
    .format("feature-service") \
    .option("gis", "myGIS") \
    .option("serviceName", service_name) \
    .option("layerName", "layer") \
    .option("tags", "dnb, prospects") \
    .option("description", "D&B Non-Residential Prospects Layer") \
    .mode("overwrite") \
    .save()

IllegalArgumentException: GIS login failed: Unable to generate token. Invalid username or password.


In [29]:
def remove_duplicate_columns(df):
    """Removes duplicate column names from a PySpark DataFrame, keeping the first occurrence."""
    cols = []
    for col_name in df.columns:
        if df.columns.count(col_name) > 1:
            # If column name appears multiple times, keep the first one
            if col_name not in [c for c in cols]:
                cols.append(col_name)
        else:
            cols.append(col_name)
    
    # Select columns by positional index to resolve duplicate references
    return df.select(*[df.schema.names[i] for i, name in enumerate(df.columns) if i == df.columns.index(name)])

parcel_land_usedesc_nodups = remove_duplicate_columns(parcel_land_usedesc)


In [88]:
ATHENA_DB = "ptl_customerexp"
ATHENA_TABLE = "dnb_parcel_intersect_dc"

df_athena = final_joined_df.drop("parcel_geometry")
print(df_athena.count())
spark.sql(f"DROP TABLE IF EXISTS {ATHENA_DB}.{ATHENA_TABLE}")

# 3. Create External Table pointing to S3 location
spark.sql(f"""
    CREATE TABLE {ATHENA_DB}.{ATHENA_TABLE}
    USING parquet
    LOCATION '{S3_OUTPUT_PATH}'    
""")

print(f"Wrote Athena table: {ATHENA_DB}.{ATHENA_TABLE}")
spark.sql(f"SELECT COUNT(*) AS row_count FROM {ATHENA_DB}.{ATHENA_TABLE}").show()

4196


In [ ]:
'''Ranking Logic Strategy
When a D&B point hits multiple overlapping parcel polygons, rank them by evaluating four attributes in order:

Commercial/Business  (lbcs_activ): D&B records represent commercial entities. Prioritize non-residential/commercial land uses (e.g., 3000.0 Industrial/Commercial) over residential (1100.0 Household) or NULL codes.
Stacked Parcel Neutrality: If a point sits in a condo stack (ll_stack_u is populated), treat non-stacked base parcels (NULL ll_stack_u) with higher priority unless matching unit-level business activity.
Deterministic Tie-Breaking: Use ll_uuid as a final tie-breaker to prevent non-deterministic partition sorts across distributed Spark tasks.
'''